# Phase 3: Filtering and Cleaning

**Purpose:**
1. Filter to the analytic sample: children ages 6–17 with valid asthma response (MCQ010 ∈ {1, 2})
2. Drop the participant identifier (SEQN) and leaky asthma variables (MCQ025, MCQ035, MCQ040, MCQ050, MCQ051)
3. Drop columns that are 100% missing (no data in any cycle)
4. Drop columns with ≥50% missingness
5. Drop spirometry quality flags, interview metadata, body measure status codes, and age-restricted variables that don't apply to the 6–17 age group

**Inputs:** `../data/processed/02_recoded.parquet`

**Output:** `../data/processed/03_cleaned.parquet`

**Note on leaky variables:** MCQ025/035/040/050/051 are downstream consequences of having asthma (e.g., "still have asthma," "had attack in past year," "ER visit for asthma"). Including them would let the model trivially predict the outcome.

In [1]:
"""
Load the recoded dataset from Phase 2.
"""

from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DIR = Path("../data/processed")

recode_df = pd.read_parquet(PROCESSED_DIR / "02b_harmonized.parquet")   # harmonized cross-cycle inputs

print(f"Loaded shape: {recode_df.shape[0]:,} rows × {recode_df.shape[1]} columns")

Loaded shape: 30,442 rows × 477 columns


In [2]:
import pandas as pd

# Assume 'recode_df' is your initial DataFrame

print(f"Original data frame shape: {recode_df.shape}")

# --- Step 1: Filter Rows ---
print("\nFiltering rows for participants between 6 and 17 years old with valid asthma status...")
df_cleaned = recode_df[(recode_df["RIDAGEYR"] >= 6) & (recode_df["RIDAGEYR"] < 18)].copy()

valid_responses = [1.0, 2.0]
df_cleaned = df_cleaned[df_cleaned['MCQ010'].isin(valid_responses)].copy()
print(f"Shape after filtering rows: {df_cleaned.shape}")


# --- Step 2: Drop Known Unwanted Columns (Knowledge-Based) ---
print("\nDropping identifier (SEQN) and data-leaky variables...")
cols_to_drop = [
    "SEQN",
    "MCQ025", "MCQ035", "MCQ040", "MCQ050", "MCQ051",
    # superseded by harmonized columns (NHANES renamed these in 2011-2012)
    "DMDBORN2", "DMDBORN4", "DMDHRBR2", "DMDHRBR4",
    "RIDAGEEX", "RIDEXAGM", "RIDEXAGY",
    # no cross-cycle coverage / redundant with RIDAGEYR
    "RIDAGEMN", "DMDSCHOL", "BMXTRI", "BMXSUB",
    # celiac items absent in 2007-2008 (drop to avoid cycle-confounded missingness; optional)
    "MCQ082", "MCQ086",
    # survey-design / administrative variables - never predictors (keep WTMEC2YR as the weight)
    "WTINT2YR", "SDMVPSU", "SDMVSTRA",
]

existing_cols_to_drop = [col for col in cols_to_drop if col in df_cleaned.columns]
df_cleaned.drop(columns=existing_cols_to_drop, inplace=True)
print(f"Shape after dropping leaky variables and ID: {df_cleaned.shape}")


# --- Step 3: Drop Columns Based on Missingness (Data-Driven) ---
print("\nDropping columns that are 100% missing...")
df_cleaned.dropna(axis=1, how='all', inplace=True)
print(f"Shape after dropping fully empty columns: {df_cleaned.shape}")

print("\nDropping columns with 50% or more missing values...")
min_non_missing = len(df_cleaned) / 2
df_cleaned.dropna(axis='columns', thresh=min_non_missing, inplace=True)
print(f"Shape after dropping variables with 50% or more missing: {df_cleaned.shape}")

print("\nDropping Spirometry Quality Variables...")
Spirometry_Quality_Variables = ['SPXNQEFF', 'SPXBQEFF', 'SPXNQFV1', 'SPXBQFV1', 'SPXBQFVC', 'SPXNQFVC']
existing_spirometry = [col for col in Spirometry_Quality_Variables if col in df_cleaned.columns]
df_cleaned.drop(columns=existing_spirometry, inplace=True)
print(f"Shape after dropping spirometry quality variables: {df_cleaned.shape}")


print("\nDropping Interview Related Variables...")
Interview_Related_Variables = ['MIAINTRP', 'MIALANG', 'MIAPROXY', 'FIAINTRP', 'RIDEXMON', 'FIAPROXY', 
                                'SIAPROXY', 'SIAINTRP', 'SIALANG', 'RIDSTATR', 'SDDSRVYR']
existing_interview = [col for col in Interview_Related_Variables if col in df_cleaned.columns]
df_cleaned.drop(columns=existing_interview, inplace=True)
print(f"Shape after dropping interview related variables: {df_cleaned.shape}")


print("\nDropping Body Measures Component Status Code and Source of Health Status Data...")
Health_Status_Data = ['BMDSTATS', 'HSAQUEX']
existing_health = [col for col in Health_Status_Data if col in df_cleaned.columns]
df_cleaned.drop(columns=existing_health, inplace=True)
print(f"Shape after dropping Body Measures Component Status Code and Source of Health Status Data: {df_cleaned.shape}")


print("\nDropping Age-Restricted Variables...")
Age_Restructed_Variables = ['ECQ020', 'ECQ080', 'ECQ090', 'WHQ030E', 'MCQ080E', 'ECQ150', 'ECD010', 'ECD070A', 'ECD070B', 'FSD670ZC', 'FSQ690', 'FSD680', 'FSD675']
existing_age_restricted = [col for col in Age_Restructed_Variables if col in df_cleaned.columns]
df_cleaned.drop(columns=existing_age_restricted, inplace=True)
print(f"Shape after dropping age-restricted variables: {df_cleaned.shape}")

# --- Final Output ---
print(f"\nFinal shape of the cleaned dataset: {df_cleaned.shape}")

Original data frame shape: (30442, 477)

Filtering rows for participants between 6 and 17 years old with valid asthma status...
Shape after filtering rows: (6784, 477)

Dropping identifier (SEQN) and data-leaky variables...
Shape after dropping leaky variables and ID: (6784, 455)

Dropping columns that are 100% missing...
Shape after dropping fully empty columns: (6784, 290)

Dropping columns with 50% or more missing values...
Shape after dropping variables with 50% or more missing: (6784, 119)

Dropping Spirometry Quality Variables...
Shape after dropping spirometry quality variables: (6784, 113)

Dropping Interview Related Variables...
Shape after dropping interview related variables: (6784, 102)

Dropping Body Measures Component Status Code and Source of Health Status Data...
Shape after dropping Body Measures Component Status Code and Source of Health Status Data: (6784, 100)

Dropping Age-Restricted Variables...
Shape after dropping age-restricted variables: (6784, 90)

Final shap

In [3]:
"""
Save the cleaned dataset for use in Phase 4 (modeling).
"""

output_path = PROCESSED_DIR / "03_cleaned.parquet"
df_cleaned.to_parquet(output_path, index=False)

size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"Saved: {output_path}")
print(f"Size:  {size_mb:.1f} MB")
print(f"Shape: {df_cleaned.shape}")
print()
print("Sample of remaining columns (first 30):")
print(list(df_cleaned.columns[:30]))
print()
print("Outcome variable (MCQ010) distribution:")
print(df_cleaned['MCQ010'].value_counts(dropna=False).to_string())

Saved: ..\data\processed\03_cleaned.parquet
Size:  0.5 MB
Shape: (6784, 90)

Sample of remaining columns (first 30):
['RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDCITZN', 'DMDEDUC3', 'DMDHHSIZ', 'DMDFMSIZ', 'INDHHIN2', 'INDFMIN2', 'INDFMPIR', 'DMDHRGND', 'DMDHRAGE', 'DMDHREDU', 'DMDHRMAR', 'DMDHSEDU', 'FIALANG', 'WTMEC2YR', 'BMXWT', 'BMXHT', 'BMXBMI', 'BMXLEG', 'BMXARML', 'BMXARMC', 'BMXWAIST', 'MCQ010', 'MCQ053', 'MCQ092', 'MCQ140', 'MCQ300B', 'ENQ010']

Outcome variable (MCQ010) distribution:
MCQ010
2.0    5524
1.0    1260
